# 05 – Statistical Analysis (EVISION 360)

EDA (Phase 04) එකේ දැක්කේ **සම්බන්ධතා**. මේ notebook එකෙන් ඒවා statistically පරීක්ෂා කරනවා.

**ප්‍රශ්නය:** රටක ආර්ථික සහ ජනගහන සාධක (GDP per capita, urbanization, population) EV adoption (නව car sales වල EV share) එක්ක කොච්චර සම්බන්ධද? ඒ සම්බන්ධය කාලයත් එක්ක ස්ථායීද? ඒ සාධක වලින් නොදුටු රටක adoption අනුමාන කරන්න පුළුවන්ද?

| Section | කරන දේ |
| --- | --- |
| 1 | Sample සහ distribution (transform එක සාධාරණද?) |
| 2 | Spearman correlations + bootstrap confidence intervals + Holm correction |
| 3 | Regression (logit scale, robust SE) + diagnostics + influential countries + robustness |
| 4 | Out-of-sample validation: leave-one-country-out |
| 5 | කාලයත් එක්ක ස්ථායීද? (වසරින් වසර + pooled panel) |
| 6 | Hypothesis tests (GDP tertiles) |
| 7 | Sri Lanka: model එකේ සීමාවන් පෙන්වන උදාහරණය |
| 8 | Findings (data වලින්ම ගණනය වෙන) |

**පරීක්ෂා කරන්නේ associations මිස causes නෙවෙයි.** Policy, ඉන්ධන මිල, vehicle supply, charging network වැනි සාධක මේ models වල නැහැ. IEA share values round කරලා තියෙනවා (2 significant figures).

In [2]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sqlalchemy import text
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src/ingestion"))
from db import get_engine

engine = get_engine()
FIG = ROOT / "reports/figures"
FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
RNG = np.random.default_rng(42)  # fixed seed: bootstrap results are reproducible

BG, PANEL, TXT, GRID = "#0b1426", "#0f1d36", "#e6f1ff", "#22345a"
CYAN, GREEN, AMBER, PINK, GREY = "#00e5ff", "#39ff88", "#ffb347", "#ff5ea8", "#8fa3c7"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": PANEL, "savefig.facecolor": BG, "text.color": TXT,
    "axes.labelcolor": TXT, "axes.edgecolor": GRID, "xtick.color": TXT, "ytick.color": TXT,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.alpha": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "axes.titlesize": 12,
    "axes.titleweight": "bold", "legend.facecolor": PANEL, "legend.edgecolor": GRID,
})


def q(sql, params=None):
    with engine.connect() as conn:
        conn.exec_driver_sql("SET search_path TO evision")
        return pd.read_sql(text(sql), conn, params=params)


def save(fig, name):
    fig.savefig(FIG / name, dpi=150, bbox_inches="tight")
    print("saved", name)


# Country-year panel: IEA EV share of new car sales (Historical, impossible values excluded) + World Bank indicators
panel = q("""
    SELECT g.geo_code, g.geo_name, f.year, f.value AS ev_share_pct,
           e.gdp_per_capita_usd, e.urban_pop_pct, e.population, e.inflation_pct
    FROM fact_ev_market_annual f
    JOIN dim_geography g USING (geo_key)
    JOIN dim_indicator i USING (indicator_key)
    JOIN dim_scenario s USING (scenario_key)
    JOIN fact_economic_indicators e ON e.geo_key = f.geo_key AND e.year = f.year
    WHERE g.geo_type = 'country' AND i.parameter_code = 'ev_sales_share' AND i.vehicle_mode = 'Cars'
      AND i.powertrain = 'EV' AND s.scenario_code = 'Historical' AND NOT f.is_suspect_value
      AND f.year BETWEEN 2015 AND 2024""")
p = (panel["ev_share_pct"] / 100).clip(0.001, 0.999)       # keep logit finite for tiny/huge shares
panel["logit"] = np.log(p / (1 - p))
panel["log_gdp"] = np.log10(panel["gdp_per_capita_usd"])   # per 10x GDP
panel["log_pop"] = np.log10(panel["population"])           # per 10x population
x = panel[panel["year"] == 2024].dropna(subset=["log_gdp", "urban_pop_pct", "log_pop"]).reset_index(drop=True)
print(f"Panel rows: {len(panel)}, countries: {panel['geo_code'].nunique()} | 2024 cross-section: n = {len(x)}")

ModuleNotFoundError: No module named 'statsmodels'

## 1. Sample සහ distribution

EV share එක 0–100% අතර bounded සහ දකුණට skewed. Linear regression එකකට කෙලින්ම දාන්න එපා, ඒ නිසා **logit transform** එකක් (log-odds) භාවිතා කරනවා. ඒකෙන් skew අඩු වෙනවාද බලමු.

In [ ]:
skew_raw, skew_logit = x["ev_share_pct"].skew(), x["logit"].skew()
print(f"Skewness of EV share (2024): raw {skew_raw:.2f} -> logit {skew_logit:.2f}")
print(f"Share range: {x['ev_share_pct'].min():.1f}% - {x['ev_share_pct'].max():.1f}% | n = {len(x)}")
print("Smallest urban share in the sample:", x.loc[x["urban_pop_pct"].idxmin(), "geo_name"], round(x["urban_pop_pct"].min(), 1), "%")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].hist(x["ev_share_pct"], bins=15, color=CYAN, edgecolor=BG); axes[0].set(title=f"EV share (skew {skew_raw:.1f})", xlabel="%")
axes[1].hist(x["logit"], bins=15, color=GREEN, edgecolor=BG); axes[1].set(title=f"logit(EV share) (skew {skew_logit:.1f})", xlabel="log-odds")
plt.tight_layout(); save(fig, "08_share_distribution.png"); plt.show()

## 2. Correlations (Spearman) + bootstrap confidence intervals

Spearman rank correlation outliers වලට ශක්තිමත්. **Bootstrap** (රටවල් නැවත නැවත sample කිරීම, 3,000 වතාවක්) මගින් 95% confidence interval එකක් ගන්නවා. Variables 4ක් එකවර test කරන නිසා p-values **Holm correction** එකෙන් සකසනවා.

In [ ]:
variables = {"log_gdp": "GDP per capita (log10)", "urban_pop_pct": "Urban population %",
             "log_pop": "Population (log10)", "inflation_pct": "Inflation %"}
rows = []
for var, label in variables.items():
    rho, pval = stats.spearmanr(x["ev_share_pct"], x[var])
    boots = []
    for _ in range(3000):
        idx = RNG.integers(0, len(x), len(x))
        boots.append(stats.spearmanr(x["ev_share_pct"].iloc[idx], x[var].iloc[idx])[0])
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    rows.append({"variable": label, "rho": rho, "ci_low": lo, "ci_high": hi, "p": pval})
corr = pd.DataFrame(rows)
corr["p_holm"] = multipletests(corr["p"], method="holm")[1]
corr["significant_5pct"] = corr["p_holm"] < 0.05
print(corr.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.6))
yy = np.arange(len(corr))[::-1]
ax.hlines(yy, corr["ci_low"], corr["ci_high"], color=GREY, lw=3)
ax.scatter(corr["rho"], yy, color=[GREEN if s else AMBER for s in corr["significant_5pct"]], s=70, zorder=3)
ax.axvline(0, color=TXT, lw=0.8)
ax.set_yticks(yy); ax.set_yticklabels(corr["variable"])
ax.set(title="Spearman rho with EV share, 2024 (95% bootstrap CI; green = significant after Holm)", xlabel="rho")
save(fig, "09_correlation_forest.png"); plt.show()

## 3. Regression (logit scale)

`logit(EV share) ~ log10(GDP per capita) + urban % + log10(population)`

- Robust (HC3) standard errors: n කුඩා නිසා heteroskedasticity ට ශක්තිමත්.
- Coefficients **odds ratios** බවට හරවනවා: EV share එකේ odds (share / (100 − share)) වල ගුණකය.
- Diagnostics: multicollinearity (VIF), influential countries (Cook's distance), residuals.
- Robustness: fractional logit (GLM) සහ influential රටවල් අයින් කිරීම.

In [ ]:
FORMULA = "logit ~ log_gdp + urban_pop_pct + log_pop"
ols = smf.ols(FORMULA, data=x).fit(cov_type="HC3")
ci = ols.conf_int()
scale = {"log_gdp": 1, "urban_pop_pct": 10, "log_pop": 1}   # odds ratio per 10x GDP, +10 pp urban, 10x population
labels = {"log_gdp": "GDP per capita, 10x", "urban_pop_pct": "Urban share, +10 pp", "log_pop": "Population, 10x"}
tab = pd.DataFrame({
    "effect": [labels[k] for k in scale],
    "odds_ratio": [np.exp(ols.params[k] * s) for k, s in scale.items()],
    "or_ci_low": [np.exp(ci.loc[k, 0] * s) for k, s in scale.items()],
    "or_ci_high": [np.exp(ci.loc[k, 1] * s) for k, s in scale.items()],
    "p_value": [ols.pvalues[k] for k in scale],
})
print(f"n = {int(ols.nobs)} | R2 = {ols.rsquared:.3f} | adjusted R2 = {ols.rsquared_adj:.3f}")
print(tab.round(3).to_string(index=False))

X = sm.add_constant(x[["log_gdp", "urban_pop_pct", "log_pop"]])
vif = {c: round(variance_inflation_factor(X.values, i), 2) for i, c in enumerate(X.columns) if c != "const"}
print("\nVIF (above ~5 would signal multicollinearity):", vif)

cooks = pd.Series(ols.get_influence().cooks_distance[0], index=x["geo_name"])
influential = cooks[cooks > 4 / len(x)].sort_values(ascending=False)
print("Influential countries (Cook's D > 4/n):", influential.round(2).to_dict())

In [ ]:
# Robustness 1: fractional logit (GLM, binomial family, robust SE) on the raw share
glm = smf.glm("I(ev_share_pct / 100) ~ log_gdp + urban_pop_pct + log_pop", data=x,
              family=sm.families.Binomial()).fit(cov_type="HC1")
# Robustness 2: drop the most influential countries and refit
drop = list(influential.index[:3])
ols_drop = smf.ols(FORMULA, data=x[~x["geo_name"].isin(drop)]).fit(cov_type="HC3")
rob = pd.DataFrame({
    "OLS logit (all)": ols.params, "p": ols.pvalues,
    "Fractional logit": glm.params, "p ": glm.pvalues,
    f"OLS, without {', '.join(drop)}": ols_drop.params, "p  ": ols_drop.pvalues,
}).drop(index="Intercept")
print(rob.round(3).to_string())
print(f"\nR2 without {', '.join(drop)}: {ols_drop.rsquared:.3f} (n = {int(ols_drop.nobs)})")

**Influential රටවල් ගැන:** Table එකේ තුන්වෙනි කොටසේ (influential රටවල් අයින් කළ පසු) coefficients, සම්පූර්ණ sample එකේ ඒවායින් වෙනස් නම්, "GDP ට සම්බන්ධයක් නැහැ" කියන නිගමනය ඒ රටවල් කිහිපයකට සංවේදියි. ඒ රටවල් (ඔවුන්ගේ ආදායම් මට්ටමට වඩා ඉහළ adoption ඇති) වෙනම අධ්‍යයනය කළ යුතු "over-performers" වෙන්න පුළුවන්.

**Coefficients කියවන විදිහ:** odds ratio 1 ට වඩා වැඩි නම් ධනාත්මක සම්බන්ධතාවක්. Confidence interval එක 1 ඇතුළත් නම් (සහ p > 0.05) ඒ සම්බන්ධය මේ sample එකෙන් වෙන් කරලා පෙන්වන්න බැහැ. R² කියන්නේ model එකෙන් පැහැදිලි වෙන variation ප්‍රමාණය.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fitted, resid = ols.fittedvalues, ols.resid
axes[0].scatter(fitted, resid, color=CYAN, alpha=0.8)
axes[0].axhline(0, color=TXT, lw=0.8)
for i in np.argsort(-np.abs(resid.values))[:4]:
    axes[0].annotate(x.loc[i, "geo_name"], (fitted[i], resid[i]), fontsize=8, xytext=(4, 4), textcoords="offset points")
axes[0].set(title="Residuals vs fitted (logit scale)", xlabel="fitted", ylabel="residual")
(osm, osr), (slope, intercept, r) = stats.probplot(resid, dist="norm")
axes[1].scatter(osm, osr, color=GREEN, alpha=0.8); axes[1].plot(osm, slope * np.array(osm) + intercept, color=AMBER)
axes[1].set(title="Normal Q-Q plot of residuals", xlabel="theoretical quantiles", ylabel="sample quantiles")
plt.tight_layout(); save(fig, "10_residual_diagnostics.png"); plt.show()
print("Shapiro-Wilk normality test on residuals: p =", round(stats.shapiro(resid)[1], 3))

## 4. Out-of-sample validation: leave-one-country-out

Model එක **in-sample** හොඳ වුණාට වැඩක් නැහැ. රටවල් 60 පමණයි, ඒ නිසා **leave-one-country-out**: එක රටක් අයින් කරලා ඉතිරි රටවල් 59 වලින් model එක හදලා, අයින් කරපු රටේ share එක අනුමාන කරනවා. (මේක cross-section එකක් නිසා එක row = එක රට, ඒ කියන්නේ country-grouped validation එකම තමයි.) Models තුනක් සසඳනවා: constant (සාමාන්‍ය), GDP පමණක්, සම්පූර්ණ model.

In [ ]:
def loo_predictions(formula):
    preds = []
    for i in range(len(x)):
        train, test = x.drop(index=i), x.iloc[[i]]
        fit = smf.ols(formula, data=train).fit()
        preds.append(100 / (1 + np.exp(-fit.predict(test).iloc[0])))   # back to share %
    return np.array(preds)

models = {"Constant (average only)": "logit ~ 1", "GDP only": "logit ~ log_gdp", "GDP + urban + population": FORMULA}
res, preds = [], {}
for name, f in models.items():
    pr = preds[name] = loo_predictions(f)
    err = pr - x["ev_share_pct"].values
    res.append({"model": name, "RMSE_pp": np.sqrt(np.mean(err ** 2)), "MAE_pp": np.mean(np.abs(err))})
loo = pd.DataFrame(res)
base = loo.loc[0, "RMSE_pp"]
loo["RMSE_gain_vs_constant_pct"] = (1 - loo["RMSE_pp"] / base) * 100
print(loo.round(1).to_string(index=False))
print("(pp = percentage points of EV share; the constant model predicts the same value for every country)")

best = preds["GDP only"]
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(x["ev_share_pct"], best, color=CYAN, alpha=0.8)
lim = max(x["ev_share_pct"].max(), best.max()) + 5
ax.plot([0, lim], [0, lim], color=AMBER, ls="--", lw=1)
for i in np.argsort(-np.abs(best - x["ev_share_pct"].values))[:5]:
    ax.annotate(x.loc[i, "geo_name"], (x.loc[i, "ev_share_pct"], best[i]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set(title="Leave-one-country-out predictions (GDP-only model)", xlabel="actual EV share (%)", ylabel="predicted EV share (%)")
save(fig, "11_loo_predictions.png"); plt.show()

## 5. කාලයත් එක්ක ස්ථායීද?

2024 එකේ ප්‍රතිඵල අනිත් වසරවලත් එහෙමමද? පියවර තුනක්:

1. 2018–2024 හැම වසරකටම එකම model එක fit කරලා GDP coefficient එක (සම්පූර්ණ sample සහ influential රටවල් අයින් කළ sample) බලනවා.
2. **Formal trend test**: GDP × වසර interaction (වසර fixed effects සහ රටවල් අනුව clustered SE). Slope එක කාලයත් එක්ක වෙනස් වෙනවාද?
3. 2018 සහ 2024 අතර slope වෙනස සෘජුවම test කිරීම.

Confidence intervals අතිච්ඡාදනය වෙනවා කියලා වෙනසක් නැහැ කියන්න බැහැ, ඒ නිසා formal test එකක් අවශ්‍යයි.

In [ ]:
def gdp_by_year(data):
    out = []
    for y in range(2018, 2025):
        z = data[data["year"] == y].dropna(subset=["log_gdp", "urban_pop_pct", "log_pop"])
        fit = smf.ols(FORMULA, data=z).fit(cov_type="HC3")
        c = fit.conf_int().loc["log_gdp"]
        out.append({"year": y, "n": len(z), "gdp_coef": fit.params["log_gdp"], "ci_low": c[0], "ci_high": c[1], "R2": fit.rsquared})
    return pd.DataFrame(out)

by_year = gdp_by_year(panel)
by_year_excl = gdp_by_year(panel[~panel["geo_name"].isin(drop)])
print("GDP coefficient by year, all countries:"); print(by_year.round(2).to_string(index=False))
print(f"Without {', '.join(drop)}:"); print(by_year_excl.round(2).to_string(index=False))

pn = panel[panel["year"].between(2018, 2024)].dropna(subset=["log_gdp", "urban_pop_pct", "log_pop"]).copy()
pn["year_c"] = pn["year"] - 2018
TREND = FORMULA.replace("log_gdp", "log_gdp + log_gdp:year_c") + " + C(year)"


def trend_fit(data):
    return smf.ols(TREND, data=data).fit(cov_type="cluster", cov_kwds={"groups": data["geo_code"]})


t_all, t_excl = trend_fit(pn), trend_fit(pn[~pn["geo_name"].isin(drop)])
trend = pd.DataFrame({
    "sample": ["all countries", f"without {', '.join(drop)}"],
    "slope_change_per_year": [t.params["log_gdp:year_c"] for t in (t_all, t_excl)],
    "ci_low": [t.conf_int().loc["log_gdp:year_c", 0] for t in (t_all, t_excl)],
    "ci_high": [t.conf_int().loc["log_gdp:year_c", 1] for t in (t_all, t_excl)],
    "p": [t.pvalues["log_gdp:year_c"] for t in (t_all, t_excl)],
})
print("Trend test (change in the GDP coefficient per year):"); print(trend.round(3).to_string(index=False))

two = pn[pn["year"].isin([2018, 2024])].copy(); two["is24"] = (two["year"] == 2024).astype(int)
diff = smf.ols("logit ~ log_gdp * is24 + urban_pop_pct + log_pop", data=two).fit(cov_type="cluster", cov_kwds={"groups": two["geo_code"]})
print(f"2018 vs 2024 difference in the GDP slope: {diff.params['log_gdp:is24']:.2f} (p = {diff.pvalues['log_gdp:is24']:.3f})")

pooled = smf.ols(FORMULA + " + C(year)", data=pn).fit(cov_type="cluster", cov_kwds={"groups": pn["geo_code"]})
print(f"Pooled constant-slope model (average over 2018-2024): GDP coefficient {pooled.params['log_gdp']:.2f}, p = {pooled.pvalues['log_gdp']:.3f} "
      f"({int(pooled.nobs)} country-years, {pn['geo_code'].nunique()} countries)")

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.fill_between(by_year["year"], by_year["ci_low"], by_year["ci_high"], color=CYAN, alpha=0.2)
ax.plot(by_year["year"], by_year["gdp_coef"], color=CYAN, marker="o", label="all countries (95% CI)")
ax.fill_between(by_year_excl["year"], by_year_excl["ci_low"], by_year_excl["ci_high"], color=PINK, alpha=0.15)
ax.plot(by_year_excl["year"], by_year_excl["gdp_coef"], color=PINK, marker="s", ls="--", label=f"without {', '.join(drop)}")
ax.axhline(0, color=TXT, lw=0.8)
ax.set(title="GDP coefficient (logit per 10x GDP), year by year", xlabel="", ylabel="coefficient"); ax.legend(loc="upper right")
save(fig, "12_gdp_coefficient_by_year.png"); plt.show()

**අර්ථකථනය:** Trend test එකේ p-value එක 0.05 ට අඩු නම් GDP සම්බන්ධය කාලයත් එක්ක වෙනස් වෙනවා කියලා statistical සාක්ෂියක් තියෙනවා. **ඒකට හේතුව** (උදා: adoption ආදායම් අඩු රටවලටත් පැතිරීම, sample එකට නව රටවල් එකතු වීම) මේ notebook එකේ පරීක්ෂා කරලා නැහැ, ඒ නිසා ඒවා උපකල්පන පමණයි. Pooled constant-slope model එක අනුමාන කරන්නේ slope එක කාලයත් එක්ක වෙනස් නොවන බවයි, trend test එකට එකඟ නැත්නම් pooled coefficient එක "සාමාන්‍යයක්" ලෙස පමණක් සලකන්න.

## 6. Hypothesis tests: GDP tertiles

රටවල් GDP per capita අනුව සමාන කණ්ඩායම් 3කට (low / middle / high) බෙදලා EV share එක සසඳනවා.

- **Kruskal–Wallis**: කණ්ඩායම් 3 අතර වෙනසක් තියෙනවාද? (non-parametric, outliers වලට ශක්තිමත්)
- **Pairwise Mann–Whitney** + Holm correction
- **Cliff's delta**: effect size (−1 සිට +1 දක්වා; 0 = වෙනසක් නැහැ)

In [ ]:
x["gdp_group"] = pd.qcut(x["gdp_per_capita_usd"], 3, labels=["low", "middle", "high"])
groups = {g: d["ev_share_pct"].values for g, d in x.groupby("gdp_group", observed=True)}
summary = x.groupby("gdp_group", observed=True).agg(
    n=("ev_share_pct", "size"), gdp_min=("gdp_per_capita_usd", "min"), gdp_max=("gdp_per_capita_usd", "max"),
    median_share=("ev_share_pct", "median"), q1=("ev_share_pct", lambda s: s.quantile(0.25)), q3=("ev_share_pct", lambda s: s.quantile(0.75)))
print(summary.round(1).to_string())
H, p_kw = stats.kruskal(*groups.values())
print(f"\nKruskal-Wallis: H = {H:.1f}, p = {p_kw:.2g}")

pairs = [("low", "middle"), ("middle", "high"), ("low", "high")]
prow = []
for a, b in pairs:
    U, pv = stats.mannwhitneyu(groups[a], groups[b], alternative="two-sided")
    prow.append({"comparison": f"{a} vs {b}", "p": pv, "cliffs_delta": 2 * U / (len(groups[a]) * len(groups[b])) - 1})
pw = pd.DataFrame(prow)
pw["p_holm"] = multipletests(pw["p"], method="holm")[1]
print(pw.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4.4))
for k, (g, vals) in enumerate(groups.items()):
    ax.scatter(np.full(len(vals), k) + RNG.uniform(-0.12, 0.12, len(vals)), vals, color=[CYAN, GREEN, AMBER][k], alpha=0.8)
    ax.hlines(np.median(vals), k - 0.25, k + 0.25, color=TXT, lw=2)
ax.set_xticks(range(3)); ax.set_xticklabels([f"{g}{chr(10)}n={len(v)}" for g, v in groups.items()])
ax.set(title="EV share by GDP-per-capita tertile, 2024 (white line = median)", ylabel="EV share (%)")
save(fig, "13_share_by_gdp_tertile.png"); plt.show()

## 7. Sri Lanka: model එකේ සීමාවන් පෙන්වන උදාහරණය

මේක **forecast එකක් නෙවෙයි.** GDP-only model එකෙන් Sri Lanka ගේ GDP per capita එක සහිත රටක share එක සඳහා **95% prediction interval** එක බලනවා. ඒකේ අරමුණ interval එක කොච්චර පළල්ද (model එකේ අවිනිශ්චිතතාව) කියලා පෙන්වීමයි. Sri Lanka ගේ policy (ආනයන බදු, ආදිය) model එකේ නැහැ.

Residuals normal distribution එකකට හොඳින් ගැලපෙන්නේ නැති නිසා (Section 3 Shapiro-Wilk), interval එක **approximate** එකක් පමණයි.

**Sri Lanka ගේ urban share එක sample එකේ පරාසයෙන් පිටත** නම්, සම්පූර්ණ model එක Sri Lanka ට යෙදීම extrapolation එකක්. එහෙම නම් GDP-only model එකම පාවිච්චි කරනවා.

In [ ]:
lka = q("""SELECT e.gdp_per_capita_usd, e.urban_pop_pct, e.population FROM fact_economic_indicators e
           JOIN dim_geography g USING (geo_key) WHERE g.geo_code = 'LKA' AND e.year = 2024""").iloc[0]
urban_lo, urban_hi = x["urban_pop_pct"].min(), x["urban_pop_pct"].max()
inside = urban_lo <= lka["urban_pop_pct"] <= urban_hi
print(f"Sri Lanka 2024: GDP per capita ${lka['gdp_per_capita_usd']:,.0f}, urban {lka['urban_pop_pct']:.1f}% "
      f"(sample urban range {urban_lo:.1f}% - {urban_hi:.1f}%) -> {'inside' if inside else 'OUTSIDE'} the observed range")

m1 = smf.ols("logit ~ log_gdp", data=x).fit()      # classical OLS: prediction intervals assume constant error variance
sigmoid = lambda v: 100 / (1 + np.exp(-v))
grid = pd.DataFrame({"log_gdp": np.linspace(x["log_gdp"].min(), x["log_gdp"].max(), 120)})
band = m1.get_prediction(grid).summary_frame(alpha=0.05)
lka_row = m1.get_prediction(pd.DataFrame({"log_gdp": [np.log10(lka["gdp_per_capita_usd"])]})).summary_frame(alpha=0.05).iloc[0]
mid, lo, hi = sigmoid(lka_row["mean"]), sigmoid(lka_row["obs_ci_lower"]), sigmoid(lka_row["obs_ci_upper"])
print(f"GDP-only model at Sri Lanka's GDP per capita: typical EV share {mid:.1f}%, 95% prediction interval {lo:.1f}% - {hi:.1f}%")
print(f"R2 of the GDP-only model: {m1.rsquared:.2f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(10 ** grid["log_gdp"], sigmoid(band["obs_ci_lower"]), sigmoid(band["obs_ci_upper"]), color=CYAN, alpha=0.18, label="95% prediction interval")
ax.plot(10 ** grid["log_gdp"], sigmoid(band["mean"]), color=CYAN, label="model: typical EV share")
ax.scatter(x["gdp_per_capita_usd"], x["ev_share_pct"], color=GREEN, alpha=0.8, label="countries (2024)")
ax.axvline(lka["gdp_per_capita_usd"], color=AMBER, ls="--")
ax.vlines(lka["gdp_per_capita_usd"], lo, hi, color=AMBER, lw=5)
ax.annotate(f"Sri Lanka GDP per capita: model range {lo:.1f}% - {hi:.0f}%", (lka["gdp_per_capita_usd"] * 1.05, 96), color=AMBER, fontsize=9)
ax.set_xscale("log"); ax.set_ylim(0, 100)
ax.set(title="What GDP alone can (and cannot) say about EV share", xlabel="GDP per capita (US$, log scale)", ylabel="EV share (%)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3); save(fig, "14_sri_lanka_prediction_interval.png"); plt.show()

## 8. Findings (data වලින්ම ගණනය කරපු)

In [ ]:
g = corr.set_index("variable")
gdp = g.loc["GDP per capita (log10)"]
sig_terms = [labels[k] for k in scale if ols.pvalues[k] < 0.05]
sig_pairs = pw.loc[pw["p_holm"] < 0.05, "comparison"].tolist()
tr, tr_excl = trend.iloc[0], trend.iloc[1]
excl_last = by_year_excl.iloc[-1]
print(f"1. GDP per capita and EV share are positively rank-correlated (rho = {gdp['rho']:.2f}, 95% CI {gdp['ci_low']:.2f} to {gdp['ci_high']:.2f}, n = {len(x)}).")
print(f"2. All countries: GDP, urbanization and population explain {ols.rsquared * 100:.0f}% of the variation in logit EV share (adjusted R2 {ols.rsquared_adj:.2f}); "
      f"individually significant at 5%: {sig_terms if sig_terms else 'none'}. Without {', '.join(drop)}: GDP coefficient p = {ols_drop.pvalues['log_gdp']:.3f}, R2 = {ols_drop.rsquared:.2f}, "
      f"so the weak full-sample result is sensitive to those countries.")
print(f"3. Predicting unseen countries: RMSE {loo.loc[2, 'RMSE_pp']:.1f} pp vs {loo.loc[0, 'RMSE_pp']:.1f} pp for a constant (gain {loo.loc[2, 'RMSE_gain_vs_constant_pct']:.0f}%), so the structural variables add little predictive power.")
print(f"4. The GDP slope changed by {tr['slope_change_per_year']:.2f} per year (95% CI {tr['ci_low']:.2f} to {tr['ci_high']:.2f}, p = {tr['p']:.3f}); "
      f"2018 vs 2024 difference {diff.params['log_gdp:is24']:.2f} (p = {diff.pvalues['log_gdp:is24']:.3f}). Without the influential countries the change is {tr_excl['slope_change_per_year']:.2f} per year (p = {tr_excl['p']:.3f}) "
      f"and the 2024 GDP coefficient is {'still significant' if excl_last['ci_low'] > 0 else 'not significant'}. The reason for the change was not tested.")
print("5. Median EV share by GDP tertile: " + ", ".join(f"{k} {v:.1f}%" for k, v in summary["median_share"].items())
      + f" (Kruskal-Wallis p = {p_kw:.1g}); significant pairs after Holm: {sig_pairs if sig_pairs else 'none'}.")
print(f"6. Most influential country in the regression: {influential.index[0]} (Cook's D {influential.iloc[0]:.2f}).")
print(f"7. Sri Lanka's GDP level alone implies a very wide range ({lo:.1f}%-{hi:.0f}% at 95%, approximate), i.e. GDP cannot pin down adoption.")

### මේකෙන් Phase 06 සහ 07 වලට ඉගෙන ගත්ත දේ

- **ආර්ථික සාධක පමණක් adoption පැහැදිලි කරන්නේ අඩුවෙන්.** ඒ නිසා EV Readiness Index (Phase 07) එකේ weights "regression එකෙන් ආවා" කියලා සාධාරණීකරණය කරන්න එපා. Weights සහ sensitivity analysis ඒ විදිහටම documentation එකේ පෙන්වන්න ඕනේ.
- Charging infrastructure, policy, vehicle prices වැනි සාධක model එකට එකතු කළොත් ඒවායේ දායකත්වය පරීක්ෂා කරන්න පුළුවන් (දැනට charging data රටවල් 43ක් පමණයි).
- Clustering (Phase 06) එකෙන් රටවල් "ආර්ථිකයෙන් බලාපොරොත්තු වන" adoption එකට වඩා ඉහළ/පහළ ඒවා (Nepal, China වැනි) හඳුනාගන්න පුළුවන්.